# SageMaker Model Deployment

Deploy trained medical imaging model to SageMaker async endpoint for asynchronous inference.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../..'))
from utils import get_or_create_role

role = get_or_create_role()
print(f"SageMaker role: {role}")


In [ ]:
%pip install sagemaker boto3

In [ ]:
import sagemaker
import boto3
from sagemaker.pytorch import PyTorchModel
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker.predictor import Predictor
import json
import time

In [ ]:
# Model configuration
sess = sagemaker.Session()
bucket = sess.default_bucket()
# Replace with your training job's model artifact path
model_data = f"s3://{bucket}/YOUR_TRAINING_JOB/output/model.tar.gz"
image_uri = "763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.5.1-gpu-py311-cu124-ubuntu22.04-sagemaker"

# Async inference configuration
async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/async-inference/output",
    max_concurrent_invocations_per_instance=4
)

In [ ]:
endpoint_name = "medical-imaging-async-endpoint"
sm_client = boto3.client("sagemaker")
existing_endpoints = sm_client.list_endpoints()["Endpoints"]

if any(ep["EndpointName"] == endpoint_name for ep in existing_endpoints):
    print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
else:
    pytorch_model = PyTorchModel(
        model_data=model_data,
        role=role,
        source_dir=".",
        entry_point="inference.py",
        framework_version="2.5.1",
        py_version="py311",
        image_uri=image_uri,
        dependencies=["requirements.txt"]
    )
    predictor = pytorch_model.deploy(
        instance_type="ml.g5.xlarge",
        initial_instance_count=1,
        endpoint_name=endpoint_name,
        async_inference_config=async_config
    )
    print(f"Endpoint '{endpoint_name}' deployed.")

## Auto-scaling (Load Balancing)

Configure Application Auto Scaling to scale instances based on the async inference queue backlog. 
Scales between 0 and 3 instances using the `ApproximateBacklogSizePerInstance` metric.

In [ ]:
aas_client = boto3.client('application-autoscaling')
resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"

# Register scalable target: min 0 (scale-to-zero), max 3 instances
aas_client.register_scalable_target(
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=0,
    MaxCapacity=3
)
print(f"Registered scalable target: min=0, max=3")

# Target tracking policy: scale based on backlog per instance
aas_client.put_scaling_policy(
    PolicyName=f"{endpoint_name}-scaling-policy",
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 5.0,
        'CustomizedMetricSpecification': {
            'MetricName': 'ApproximateBacklogSizePerInstance',
            'Namespace': 'AWS/SageMaker',
            'Dimensions': [{'Name': 'EndpointName', 'Value': endpoint_name}],
            'Statistic': 'Average'
        },
        'ScaleInCooldown': 300,
        'ScaleOutCooldown': 60
    }
)
print(f"Applied scaling policy: target backlog=5 per instance, scale-out cooldown=60s, scale-in cooldown=300s")

In [ ]:
class MedicalImagingAsyncPredictor:
    def __init__(self, endpoint_name, bucket):
        self.endpoint_name = endpoint_name
        self.runtime_client = boto3.client('sagemaker-runtime')
        self.s3_client = boto3.client('s3')
        self.bucket = bucket
        self.labels = [
            'Surgical_implant', 'Vertebral_collapse', 'Spondylolysthesis',
            'No_finding', 'Foraminal_stenosis', 'Other_lesions',
            'Disc_space_narrowing', 'Osteophytes'
        ]
    
    def predict_async(self, s3_image_uri):
        
        payload = json.dumps({"file_path": s3_image_uri})
        input_key = "async-inference/input/request.json"
        self.s3_client.put_object(Bucket=self.bucket, Key=input_key, Body=payload)
        input_location = f"s3://{self.bucket}/{input_key}"
        response = self.runtime_client.invoke_endpoint_async(
            EndpointName=self.endpoint_name,
            InputLocation=input_location,
            ContentType="application/json"
        )
        return response['OutputLocation']
    
    def get_result(self, output_path, wait=True, timeout=600):
        bucket, key = output_path.replace("s3://", "").split("/", 1)
        
        if wait:
            start_time = time.time()
            print(f"Waiting for result at {output_path}...")
            while time.time() - start_time < timeout:
                try:
                    obj = self.s3_client.get_object(Bucket=bucket, Key=key)
                    result = json.loads(obj['Body'].read().decode('utf-8'))
                    predictions = result["predictions"][0]
                    predicted_idx = result["predicted_class"][0]
                    return {
                        "probabilities": dict(zip(self.labels, predictions)),
                        "predicted_class": self.labels[predicted_idx],
                        "confidence": result["confidence"][0]
                    }
                except self.s3_client.exceptions.NoSuchKey:
                    elapsed = int(time.time() - start_time)
                    print(f"Waiting... ({elapsed}s)", end='\r')
                    time.sleep(5)
            raise TimeoutError("Result not available within timeout")
        return output_path
    
    def delete_endpoint(self):
        sm_client = boto3.client('sagemaker')
        sm_client.delete_endpoint(EndpointName=self.endpoint_name)

In [ ]:
# Upload local sample to S3 for testing
s3_client = boto3.client('s3')
sample_key = "async-inference/samples/sample_image.dcm"
s3_client.upload_file("samples/sample_image.dcm", bucket, sample_key)
s3_image_uri = f"s3://{bucket}/{sample_key}"

# Test async inference
predictor_wrapper = MedicalImagingAsyncPredictor("medical-imaging-async-endpoint", bucket)

# Submit async request
output_path = predictor_wrapper.predict_async(s3_image_uri)
print(f"Inference submitted. Output will be at: {output_path}")

# Wait for and retrieve result
result = predictor_wrapper.get_result(output_path, wait=True)

print(f"\nPredicted: {result['predicted_class']}")
print(f"Confidence: {result['confidence']:.4f}")
print("\nAll probabilities:")
for label, prob in result['probabilities'].items():
    print(f"  {label}: {prob:.4f}")

In [ ]:
endpoint_name = "medical-imaging-async-endpoint"
sm_client = boto3.client('sagemaker')

try:
    # Deregister auto-scaling
    aas_client = boto3.client('application-autoscaling')
    resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"
    aas_client.deregister_scalable_target(
        ServiceNamespace='sagemaker',
        ResourceId=resource_id,
        ScalableDimension='sagemaker:variant:DesiredInstanceCount'
    )
    print(f"Deregistered auto-scaling for: {endpoint_name}")
except Exception as e:
    print(f"Auto-scaling cleanup (may not exist): {e}")

try:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    endpoint_config_name = desc['EndpointConfigName']
    config_desc = sm_client.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
    model_name = config_desc['ProductionVariants'][0]['ModelName']

    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted endpoint: {endpoint_name}")

    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"Deleted endpoint config: {endpoint_config_name}")

    sm_client.delete_model(ModelName=model_name)
    print(f"Deleted model: {model_name}")
except sm_client.exceptions.ClientError as e:
    print(f"Cleanup error: {e}")